# dock-vision: YOLO training on Colab

Fine-tune YOLO11 on the merged dataset (synthetic + optional public data).

**Runtime**: use a GPU (Runtime > Change runtime type > T4 GPU).

**What happens**: clones the repo, generates synthetic scenes, optionally
ingests public datasets you upload, merges, trains, evaluates, and gives
you the best weights to download.

In [ ]:
!pip install -q ultralytics opencv-python-headless

## Setup: clone the repo

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/najnaj20/dock-vision'
if not Path('/content/dock-vision').exists():
    !git clone --quiet {REPO_URL} /content/dock-vision
%cd /content/dock-vision
print('repo ready')

## Data

Step 1 generates the synthetic baseline (300 scenes, ~1 min on CPU).

Step 2 is optional: if you downloaded public datasets from Roboflow
(YOLOv8 format), upload them here and they get merged in too. Skip it by
answering `n` if you only want synthetic data.

In [ ]:
!python scripts/generate_synthetic.py --num 300
!python scripts/merge_datasets.py

In [ ]:
answer = input('Upload public dataset zips (Roboflow YOLOv8)? [y/N]: ').strip().lower()
if answer == 'y':
    from google.colab import files
    for fname in files.upload().keys():
        print('ingesting', fname)
        !python scripts/download_datasets.py --zip /content/{fname}
    !python scripts/merge_datasets.py
else:
    print('synthetic-only dataset, proceeding')

## Train

Nano is a fast baseline; switch to `yolo11s.pt` for the final model.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11n.pt')  # or yolo11s.pt
model.train(
    data='data/merged/data.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    project='/content/runs',
    name='train',
)

## Evaluate

Reports mAP50, mAP50-95, precision, recall on the held-out val split.

In [ ]:
metrics = model.val(data='data/merged/data.yaml', project='/content/runs', name='val')
print(f'mAP50:   {metrics.box.map50:.4f}')
print(f'mAP50-95: {metrics.box.map:.4f}')
print(f'Precision: {metrics.box.mp:.4f}')
print(f'Recall:    {metrics.box.mr:.4f}')

## Download weights

Save `best.pt` locally, then run inference with `scripts/predict.py`.

In [ ]:
from google.colab import files
files.download('/content/runs/train/weights/best.pt')